# Practical session 3: Selective sensing, parallel behaviors and environmental dynamics

**IMPORTANT** Session 3 still has to be updated, please do not start it yet. 
When it will be ready you will be notified that a new version is available when you will start Vivarium. You will have to proceed with the update to have access to the last version.

**TODO**: explain restart button if we add it. Also explain that if the computer goes to sleep, you need to restart.

In the last practical session, we saw how to define, attach and detach a behavior on an agent. We implemented three distinct behaviors: `slow_down`, `fear` and `aggression`.

In this section we will see more sensing abilities the agent is equipped and will define new behaviors using them and combine them together. We will also see how to make the agent consume resources that spawn in the environment. Finally, we will also see how to attach those behaviors on multiple agents interacting together within a Vivarium scene.

As usual, let's connect this notebook to the simulation:

In [2]:
from vivarium.controllers.vivarium_controller import VivariumController
controller = VivariumController.start_session(scene_name="session_3")

INFO:vivarium.controllers.vivarium_controller:Server already running with scene 'session_3'. Connecting.
INFO:vivarium.controllers.vivarium_controller:Controller thread started on client
INFO:vivarium.controllers.vivarium_controller:Jupyter was started from Panel interface; skipping interface start. If you want to start another interface, set the environment variable VIVARIUM_JUPYTER_FROM_PANEL to 0.
INFO:vivarium.controllers.vivarium_controller:Controller thread is already running
INFO:vivarium.controllers.vivarium_controller:VivariumController session 'session_3' is started


## Selectively detecting scene objects

To define a repertoire of interesting behaviors, we need the agent to selectively sense the proximity of different types of entities around them. For example, we might want to define a behavior for obstacle avoidance and another one for attraction towards mates. The first behavior will require sensors information about objects, whereas the second will require the proximeters to detect other agents (although there is only one agent in the scene for now, we'll add more at the end of this session). 

Since there is only one agent, let's create an alias variable to access it as in previous sessions:

In [10]:
agent = controller.agents[0]

As seen in the scene map on the left, there are three types of entities in the current scene: a blue square and a number of orange and green circles. By default in Vivarium, square entites represent agents and circle entities represent objects. Here we have two subtypes of objects: large orange ones and smaller green ones. In this session, we call the orange objects "obstacles" and the green objects "resources". 

Each entity in the scene, either agents or objects, is associated with a *subtype*. The subtype of the agent is `agent`, the subtype of the green objects is `resource` and the subtype of the orange objects is `obstacle`. You can list the available subtypes in the current scene with:

In [11]:
controller.subtypes

['agent', 'resource', 'obstacle']

We can filter the result returned by the agent's proximeters by providing the argument `sensed_entities` to the `proximeters` function:

In [12]:
left, right = agent.proximeters(sensed_entities=["obstacle"])
print(left, right)

0.0 0.07796132564544678


Executing the cell above will return the proximeter activations only for the entities with the `obstacle` subtype, i.e. the orange circles in the scene. 

Note that the sensed entities can be occluded by other entities, whatever their subtype. This mean that if e.g. a `resource` object is closer than any `obstacle` object in the proximiter field of view, then the cell above will return `0` for that proximiter. This is somehow similar to how our own eyes sense objects: if you look at a tree but there is a wall between the tree and yourself, you won't see that tree. 

You can move an obstacle in the field of view of the proximiters using the drag and drop method (see session 1 or 2 for how to drag and drop) and re-execute the cell above to observe the change in the returned values. You can also check that proximeters are not activated when the closest object is a resource (green circles).

The `sensed_entities` argument requires a list of strings (`["obstacle"]` in the example above). In Python, a list is a collection of values separated by commas and surrounded by square bracket: `["obstacle"]` is therefore a list of only one element (the character string `"obstacle"`), whereas `["obstacle", "agent"]` is a list of two elements (the strings `"obstacle"` and `"agent"`). 


The agent's sensors can detect multiple subtypes of entities. For instance, if we want the agent to sense both resources and obstacles:

In [13]:
left, right = agent.proximeters(sensed_entities=["resource", "obstacle"])
print(left, right)

0.0 0.07796132564544678


In that case, each proximeter will sense the closest entity with one of the indicated subtypes (i.e. here the closest entity which either a resource or an obstacle).

This function gives an error message if you provide a string that doesn't correspond to an existing subtype or that is spelled wrong, and ask you to select a type among the correct ones. 

In [14]:
agent.proximeters(sensed_entities=["ressources"])  # typos in the entity subtype

AssertionError: Please specify valid sensed entities among ['agent', 'resource', 'obstacle']

The above cell returns an alarming error message, it's normal because we intentionnaly made a type in the subtype (`"ressources"` instead of `"resource"`). In the last line of the error, the valide subtypes are indicated. Thus we can correct it:

In [15]:
agent.proximeters(sensed_entities=["resource"])  # typos in the entity subtype

[0.0, 0.0]

Using this mechanism for selectively sensing objects, we can now define behaviors targeting specific object subtypes. In this [slide from the class](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit#slide=id.g31e1b425a3_0_0), we saw how different ways of connecting proximeter activations to motor activations results in different type of behaviors. Two of these behaviors result in moving closer to a sensed object (`love` and `aggression`) while the two others result in moving away from a sensed object (`shyness` and `fear`). Among these four behaviors, two of them use excitatory connections, resulting in the agent moving faster when it is closer from the sensed object (as in `aggression` and `fear`) ; while the others use inhibitory connections, resulting in the agent moving slower when it is closer to an object (as in `love` and `shyness`).

Let's for instance define a behavior for avoiding obstacles. Which of the four behaviors illustrated in the slide is best suited for this? We want the agent to move away from the obstacle, so `shyness` and `fear` are our two candidates. We might also prefer the agent to avoid collisions with the obstacles, so moving slower when the obstacle is closer seems relevant here. We have our winner behavior: `shyness` is the best candidate for a behavior which aims at avoiding obstacles. 

As illustrated on the slide, the  `shyness` behavior consists in crossed inhibitory connections. By "crossed" we mean that each proximeter, left or right, is connected to the wheel on the **opposite** side of the agent (left-to-right, right-to-left). By "inhibitory", we mean that the more the proximeter is activated, the **less** the wheel it is connected will be activated. Since both the proximeter and motor activations are values bounded between 0 and 1, the `shyness` behavior therefore correpond to:

$$m_L = 1 - s_R$$
$$m_R = 1 - s_L$$

where $s_L$ (resp. $s_R$) corresponds to the left (resp. right) proximeter sensor activation ; and $m_L$ (resp. $m_R$) correspond to the left (resp. right) wheel motor activation. This way, the more a given proximeter is activated, the less the the opposite wheel will be activated.

Using the way to define behaviors we have seen in the previous session, the `shyness` behavior can therefore be written as:

In [16]:
# Shyness behavior

def shyness(agent):
    # First we read the sensor values
    left_sensor, right_sensor = agent.proximeters()

    # Then we compute the corresponding motor activations from the above formula
    left_motor = 1 - right_sensor
    right_motor = 1 - left_sensor

    # Finally we return the left and right motor activations, in this order
    return left_motor, right_motor
    

Now we can attach the `shyness` behavior to the agent:

In [17]:
agent.attach_behavior(shyness)

Now the agent should navigate in the scene map, avoiding any entity it encounters. Given the above definition of the `shyness` behavior, the agent move at full speed when no object is in its field of view, reduces its speed if it approaches and object and turns away from the sensed object. 

However this behavior senses all objects in the scene, i.e both green resources and orange obstacle. If we instead want to only avoid obstacles and not resources, we can use the selective sensing behavior we have seen at the start of this session. Remember the for sensing only obstacles in the scene we write:

In [18]:
agent.proximeters(sensed_entities=["obstacle"])

[0.0, 0.10153442621231079]

Therefore we can define an obstacle avoidance behavior as:

In [19]:
def obstacle_avoidance(agent):
    # First we read the sensor values, sensing only the entities with subtype "obstacle"
    left_sensor, right_sensor = agent.proximeters(sensed_entities=["obstacle"])

    # Then we compute the corresponding motor activations from the above formula
    left_motor = 1 - right_sensor
    right_motor = 1 - left_sensor

    # Finally we return the left and right motor activations, in this order
    return left_motor, right_motor
    

Now we can detach the `shyness` behavior which is currently being executed and attach the new `obstacle_avoidance` behavior we have just defined:

In [20]:
agent.detach_all_behaviors(stop_motors=True)
agent.attach_behavior(obstacle_avoidance)

The agent should now smoothly navigate between the obstacles in the scene. 

## Executing multiple behaviors in parallel

As the `obstacle_avoidance` behavior only senses the orange obstacles, the agent currently does not react to the sensing of green resources. Let's implement a behavior that makes the agent forage for resources and execute it together with the obstacle avoidance behavior.

**Q1:** Define a behavior allowing the agent to forage for resources, let's call it `foraging`. The agent has to orient itself toward resources, with a speed proportional to the proximiter activations (the closer the resource, the higher the speed) 

- *Tip 1:* First think about which of the [four behaviors we have seen in class](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit?usp=sharing) is best suited for foraging. 
- *Tip 2:* You already saw how to detect obstacles in the obstacle avoidance behavior. Here the agent will instead have to detect resources. The subtype of the resources is `"resource"`.

In [21]:
def foraging(agent):
    # your code here

    # TO REMOVE
    left, right = agent.proximeters(['resource'])
    return right, left
    

Agents can actually run several behaviors in parallel. Currently, only the `obstacle_avoidance` behavior should be attached to the agent. You can attach the `foraging` behavior you have just define in addition:

In [22]:
agent.attach_behavior(foraging)

Now both the `obstacle_avoidance` behavior we previously attached and the `foraging` behavior are executed on the agent. In consequence you should see the agent avoiding obstacles while being attracted by the resources.  We can check that two behaviors are indeed attached with:

In [23]:
agent.print_behaviors()

Attached behaviors: ['obstacle_avoidance', 'foraging'], Started behaviors: ['obstacle_avoidance', 'foraging']


When multiple behaviors are executed in parallel on the same agent, the motor activations of the wheels correspond to the average of the motor activations returned by each behavior. This averaging is implemented internally, you don't need to worry about it when you define the behaviors.

**Q2:** Considering that both the `obstacle_avoidance` and the `foraging` behaviors are currently executed in parallel on the agent, what are the values of the left and right wheel activations in situations where:

- There is no object in the agent's field of view:

*your answer here*

- There is a green resource maximally activating the left proximeter and no object sensed by the right proximeter:

*your answer here*

- There is no object sensed by the left proximeter and an orange obstacle maximally activating the right proximeter:

*your answer here*

- There is an orange obstacle maximally activating the ledt proximeter and a green resource maximally activating the right proximeter:

*your answer here*

## Environmental dynamics

For now, the scene in which the agent is evolving is quite static: the agent interacts with the existing object but there is nothing that appears or disappears in the environment. We are now going to see how we can generate resources appearing at random positions in the environment and disappearing whenever an agent consumes them. 

### Consumption mechanism

The consumption mechanism specifies how some entities can consume other entities, making them disappear from the environment. For this we need to define:

- A source subtype, specifying what subtype is consuming something. As we want agents to consume resources, the source subtype will be `"agent"`.
- A target subtype, specifying which subtype is being consumed by the source subtype. As we want agents to consume resources, the target subtype will be `"resource"`.
- A consumption range, specifying what is the distance between two entities at which the consumption is triggered. This 

Programatically we do it this way:

In [24]:
# We first specify the "source subtype" of the consumption mechanism, 
# i.e. what subtype is willing to consume something:
controller.consumption.source_subtype = "agent"

# Then we specify the "target subtype" of the consumption mechanism,
# i.e. what subtype is being consumed:
controller.consumption.target_subtype = "resource"

# Then we specify the distance range at which the consumption triggered
# Here we trigger the consumption when the center of the agent is at a distance
# less than 12.5 from the center of a resource:
controller.consumption.range = 1

# Finally we activate the consumption mechanism with:
controller.consumption.start = True

Now you should see the agent consuming resources, meaning that the resources are disappearing whenever is close to them (at a distance lesser than 1). At some point, all resources will have been consumed.

### Spawning mechanisms

The spawning mechanism enables to regularly spawn new entities in the environment. We are going to use it for regularly spawning new resources in order to avoid their depletion. For this we need to define:
- The subtype of entities that will be spawned. Here we want to spawn resources, so the subtype will be `"resource"`.
- The time interval at which resources will spawn. Below we spawn them every 300 time steps. Higher values will make resources spawn less frequently (i.e. more time between each resource spawn).

In [25]:

# We first specify what subtype will be spawning
controller.spawn.subtype = "resource"

# Then we specify the time interval at which resource will spawn
# Higher values will make resources spawn less frequently
# (i.e. more time between each resource spawn)
controller.spawn.period = 300

# Finally we activate the spawning mechanism
controller.spawn.start = True

Now you should see resources spawning at random positions in the environment at regular intervals, while the agent is consuming them. The resources will spawn until the maximum number of resources is reached in the environment (it is set to 15 in the current scene). If this maximum number is reached and the agent consumes a resource, another one will appear at a random position in the scene.

#### Controlling spawning positions

You can also control the position range where the resources will appear with the `position_range` parameter. This parameter is a list of 4 values: `(x_min, x_max, y_min, y_max)` where `x_min` and `x_max` are the minimum and maximum `x` coordinates of the spawning area, and `y_min` and `y_max` are the minimum and maximum `y` coordinates of the spawning area. Note that the current scene has a size of 100 by 100, as shown by the numbers on the x and y axis of the map. 

For example, to make resources appear only in the top-left quarter of the map (i.e. the area between x=0 and x=35, and y=75 and y=100) we write:

In [26]:
# then, start the spawning apparition in a specific area
# controller.start_resources_apparition(interval=interval, position_range=((0, 50), (100, 200)))
controller.spawn.position_range = (0, 25, 75, 100)

**Q3:** Make resources appear in the bottom-right quarter of the map:

In [27]:
# Your code here


To test your answer to the last question you might want to first remove all currently existing resources from the environment. You can achieve this with:

In [28]:
for obj in controller.objects:  # Iterate through all objects
    if obj.subtype == "resource":  # If the object is a resource
        obj.exists = False  # Mark this object as non-existing

If needed you can reuse the logic the cell above to change the attribute of entites in batch. For instance if we want to change the color of all obstacles to purple we can write:

In [30]:
for obj in controller.objects:  # Iterate through all objects
    if obj.subtype == "obstacle":  # If the object is an obstacle
        obj.color = "purple"  # Set the color of the object to purple

### Starting and stopping the consumption and spawning mechanims

We can stop the consumption mechanism with:

In [31]:
controller.consumption.start = False

Now agent are no longer consuming resources

If we want to start it again we execute:

In [32]:
controller.consumption.start = True

Similarly for the spawning mechansim we can stop it with:

In [33]:
controller.spawn.start = False

Now resources are no longer spawning in the environment. To start it again:

In [34]:
controller.spawn.start = True

## Dealing with multiple agents

This section explains how to deal with multiple agents and how to attach different behaviors to them.

At the moment there is only 1 existing agent that we can see in the scene. But there is actually another one that is not "existing" yet. We can see it with:

In [35]:
controller.agents

[<vivarium.environment.components.entities.braitenberg.controller.AgentController object at 0x13827bef0>, <vivarium.environment.components.entities.braitenberg.controller.AgentController object at 0x1382af560>]

which returns a list with two elements in it, each one corresponding to an agent (internally it corresponds to a Python object of type `AgentController`). The first element in the list is accessed with `controller.agent[0]` (in Python list indices starts at 0) and corresponds to the agent we have manipulated above (using the alias variable `agent` which was set to `controller.agent[0]` at the start of this notebook). We can check that this first agent indeed exists with:

In [36]:
controller.agents[0].exists

True

which is `True`, meaning that this agent exists (it is the one we see in the interface). Let's check it for the second agent (index `1` of the list):

In [37]:
controller.agents[1].exists

True

which is `False`, meaning that this agent does not exist yet. We can make this agent exist with:

In [38]:
controller.agents[1].exists = True

Now you should see a second agent in the interface. 

Let's rename our original `agent` to `agent_0` and create an alias `agent_1` for the second one to easily access them:

In [39]:
agent_0 = controller.agents[0]
agent_1 = controller.agents[1]

`agent_0` is the same as the one we called `agent` before.

Let's change the color of the second agent to be red, so that we can more easily distinguish it in the scene map:

In [40]:
agent_1.color = 'red'

Now you have access to the two agents through the variables `agent_0` and `agent_1` (these variables names are arbitrary, you can choose whatever you want, e.g. `predator` and `prey`). You can attach and start behaviors on each agent independently, in the same way as you did before, simply using either the `agent_0` and `agent_1` variables instead of only the `agent` one as before.

As an example, let's say we want to attach the `obstacle_avoidance` behavior we have defined above to `agent_0`, and both the `obstacle_avoidance` and the `foraging` behaviors to `agent_1`. 

In [43]:
# detach the agent_0 behaviors and only attach the obstacle_avoidance behavior
agent_0.detach_all_behaviors()
agent_0.attach_behavior(obstacle_avoidance)
print("Agent_0 behaviors:")
agent_0.print_behaviors()
print()

# attach the obstacle_avoidance and foraging behaviors to agent_1
agent_1.attach_behavior(obstacle_avoidance)
agent_1.attach_behavior(foraging)
print("Agent_1 behaviors:")
agent_1.print_behaviors()

Agent_0 behaviors:
Attached behaviors: ['obstacle_avoidance'], Started behaviors: ['obstacle_avoidance']

Agent_1 behaviors:
Attached behaviors: ['obstacle_avoidance', 'foraging'], Started behaviors: ['obstacle_avoidance', 'foraging']


Let's detach the behaviors and stop the motors of both agents with the following cell. Using this `for` loop on the `controller.agents` list and the `detach_behaviors` function enables you to detach the behaviors of all the agents at once.

In [45]:
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)

**Q4:** Let's implement a simple prey-predator interaction where:

- `agent_0` will be the predator and `agent_1` the prey:
  - `agent_0` will execute the `aggression` behavior toward `agent_1`.
  - `agent_1` will execute the `fear` behavior toward `agent_0`
- `agent_1` will forage for the green resources.
- Both agents will avoid obstacles.

You can write the corresponding code in the cells below.

First let's define the `fear` and `aggression` behaviors towards other agents in the cell below.

- *Tip 1:* You can can find the illustation of each behavior in [the slide]('https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit?usp=sharing). 
- *Tip 2:* You now want to target these behaviors toward other agents. The agents are of subtype `"agent"`.

In [46]:
def fear(agent):
    # your code here
    
    # TO REMOVE
    left, right = agent.proximeters(sensed_entities=['agent'])
    return left, right


def aggression(agent):
    # your code here
    
    # TO REMOVE
    left, right = agent.proximeters(sensed_entities=['agent'])
    return right, left
    

Modify the cell below to attach the previously defined behaviors to the agents. We also directly provide lines of code to modify the aspect of the agents (size, color) and their speed.

In [49]:
agent_0.diameter = 10.  # Increase the diameter of agent_0
agent_0.color = 'red'  # Change its color to red
# agent_0.wheel_diameter = 3.0  # Increasing the diameter of the wheels increases the maximum speed

agent_1.diameter = 4.  # Decrease the diameter of agent_0
agent_1.color = 'cyan'  # Change its color to cyan
agent_1.max_speed = 1.5   # With larger wheels, agent_1 will move faster than agent_0


# attach the aggression and obstacle_avoidance behaviors to agent_0
# your code here

agent_0.detach_all_behaviors()
agent_0.attach_behavior(aggression)
agent_0.attach_behavior(obstacle_avoidance)


# attach the fear and obstacle_avoidance behaviors to agent_1
# your code here
agent_1.detach_all_behaviors()
agent_1.attach_behavior(fear)
agent_1.attach_behavior(obstacle_avoidance)
agent_1.attach_behavior(foraging)

You should now observe a simple "prey-predator" interaction, where `agent_0` tries to catch `agent_1` while `agent_1` tries to escape from `agent_0` and forage for resources.
They should be both avoiding obstacles as well.

## Modifying the sensor fields of view

But because the sensors of `agent_1` are only directed in the forward direction, it is really hard to avoid the `agent_0` when it comes from behind it (it cannot see it in that case). Additionally, the `agent_0` is a predator but has a pretty bad vision because it can't see very far.


We can fix this by modifying the sensors characteristics of the agents, by changing their angles and max ranges by using the following cell. 

- The `prox_dist_max` parameter is the maximum distance at which the sensors can detect entities
- The `prox_cos_min` parameter is the minimum cosine of the angle between the sensor and the entity in order to detect it

So the `prox_cos_min` value is between -1 and 1. The closer this value is to 1, the narrower the sensor field of view, and inversely for -1. By default this value is 0, corresponding to a field of view covering a semi-disk (as visualized in light red in the interface).

Let's modify the field of view of the agents:

In [50]:
# increase the range of the red agent and decrease its sensor angle
agent_0.proxs_dist_max = 50.
agent_0.proxs_cos_min = 0.8

In [51]:
# decrease the range of the cyan agent and increase its angle
agent_1.proxs_dist_max = 20.
agent_1.proxs_cos_min = -0.9

In [52]:
agent_0.max_speed = 2.0

That's it for today. Don't forget to properly close the simulator before closing this notebook:

In [ ]:
controller.close_session()
# stop_server_and_interface(safe_mode=False)

Now that you finished session 3, you can now jump to the notebook of [session 4](session_4.ipynb).